# Transcriptómica espacial

## Enlaces del tutorial
Explora los siguientes tutoriales para iniciarte en el análisis transcriptómico espacial:

* Paquete de herramientas para transcriptomica espacial [semla](https://ludvigla.github.io/semla/articles/getting_started.html). Junto con este tutorial se encuentran otros de los que se deriva gran parte de este notebook, y que abarcan la extracción y el análisis de transcriptómica espacial con el paquete semla.

* [Viñeta Transcriptomica Espacial Seurat](https://satijalab.org/seurat/articles/spatial_vignette.html) ¿ Cómo realizar análisis transcriptómicos espaciales con el paquete Seurat?

# Instalar los paquetes

In [ ]:
## Función para ejecutar comandos de terminal en Google Colab con el kernel de R
shell_call <- function(command, ...) {
  result <- system(command, intern = TRUE, ...)  # Ejecuta el comando de terminal y guarda la salida
  cat(paste0(result, collapse = "\n"))  # Imprime la salida en un formato legible
}
# Descarga el código desde la URL especificada y lo guarda como "add_cranapt_jammy.sh"
download.file("https://github.com/eddelbuettel/r2u/raw/master/inst/scripts/add_cranapt_jammy.sh",
              "add_cranapt_jammy.sh")
Sys.chmod("add_cranapt_jammy.sh", "0755")
shell_call("./add_cranapt_jammy.sh")
bspm::enable()
options(bspm.version.check=FALSE)
shell_call("rm add_cranapt_jammy.sh")
# Establece un límite de tiempo más alto para evitar interrupciones durante descargas de paquetes.
options(timeout=1000)

In [ ]:
# Instala el paquete "semla" desde GitHub forzando su actualización.
remotes::install_github("ludvigla/semla", upgrade=T, force = TRUE)

In [ ]:
# Instala paquetes
cranPkgs2Install = c("BiocManager", "ggpubr", "Seurat", "hdf5r", "openxlsx",
                     "enrichR", "clustree", "DT","ggcorrplot","scatterpie", "pheatmap")
install.packages(cranPkgs2Install, ask=FALSE, update=TRUE, quietly=TRUE)

In [ ]:
biocPkgs2Install = c("SingleCellExperiment", "ReactomePA", "org.Hs.eg.db", "glmGamPoi", "fgsea", "limma")
BiocManager::install(biocPkgs2Install, ask=FALSE, update=TRUE, quietly=TRUE)

In [ ]:
# Descarga paquetes
# Suprime los mensajes al cargar paquetes para mantener la salida limpia.
suppressPackageStartupMessages({
library(Seurat)  # Marco de análisis para RNA-seq de célula única.
library(data.table)  # Manejo eficiente de grandes volúmenes de datos.
library(ggplot2)  # Visualización basada en la gramática de gráficos.
library(plotly)  # Visualizaciones interactivas.
library(RColorBrewer)  # Paletas de colores predefinidas.
library(dplyr)  # Manipulación de datos.
library(semla) # Herramientas para transcriptómica espacial.
library(clustree)  # Visualización de resoluciones de clústeres.
library(ReactomePA)  # Análisis de vías metabólicas.
library(org.Hs.eg.db)  # Base de datos de anotación génica humana.
library(ggpubr)  # Visualizaciones listas para publicación.
library(enrichR)  # Análisis de enriquecimiento génico.
library(stringr)  # Manipulación de cadenas (caracteres).
library(openxlsx)  # Visualizaciones archivos Excel.
library(patchwork)  # Combina múltiples gráficos.
library(SingleCellExperiment)  # Estructura para datos de célula única.
})

# Introducción

La transcriptómica espacial es una técnica que combina la información de expresión génica con la localización física de las células dentro de un tejido. A diferencia de los métodos tradicionales de transcriptómica, aquí no solo sabemos qué genes se expresan, sino también dónde se expresan, lo que permite estudiar la organización celular y los microambientes tisulares de manera más precisa.

En este contexto, las imágenes histológicas juegan un papel clave. Una de las más utilizadas es la tinción H&E (Hematoxilina y Eosina), que colorea los tejidos para resaltar su estructura:

* Hematoxilina tiñe los núcleos celulares de azul/púrpura.

* Eosina tiñe el citoplasma y componentes extracelulares de rosa.

La combinación de transcriptómica espacial con imágenes H&E permite vincular los perfiles moleculares con la arquitectura tisular, ofreciendo una visión integral de cómo las células se organizan y funcionan en su entorno natural.

## Descargar los datos

Los datos descargados a continuacion corresponden a un dataset de transcriptomica espacial disponible desde la pagina de 10x y generado con [Space Ranger](https://www.10xgenomics.com/support/software/space-ranger/latest), la muestra corresponde a tejido de cancer de mama.
Puedes revisar detalles en los siguientes enlaces:

- [Human Breast Cancer (Block A Section 1)](https://www.10xgenomics.com/datasets/human-breast-cancer-block-a-section-1-1-standard-1-1-0)

- [Human Breast Cancer (Block A Section 2)](https://www.10xgenomics.com/datasets/human-breast-cancer-block-a-section-2-1-standard-1-1-0)

In [ ]:
shell_call("mkdir -p ST_Exercises")

# Descarga los archivos que contienen ejercicios de transcriptómica espacial.
download.file('https://cf.10xgenomics.com/samples/spatial-exp/1.1.0/V1_Breast_Cancer_Block_A_Section_2/V1_Breast_Cancer_Block_A_Section_2_filtered_feature_bc_matrix.h5','ST_Exercises/Breast_Cancer_Block_A_Section_2_filtered_feature_bc_matrix.h5')
download.file('https://cf.10xgenomics.com/samples/spatial-exp/1.1.0/V1_Breast_Cancer_Block_A_Section_2/V1_Breast_Cancer_Block_A_Section_2_spatial.tar.gz','ST_Exercises/Breast_Cancer_Block_A_Section_2_spatial.tar.gz')
download.file('https://cf.10xgenomics.com/samples/spatial-exp/1.1.0/V1_Breast_Cancer_Block_A_Section_2/V1_Breast_Cancer_Block_A_Section_2_metrics_summary.csv','ST_Exercises/Breast_Cancer_Block_A_Section_2_metrics_summary.csv')
download.file('https://cf.10xgenomics.com/samples/spatial-exp/1.1.0/V1_Breast_Cancer_Block_A_Section_2/V1_Breast_Cancer_Block_A_Section_2_web_summary.html','ST_Exercises/Breast_Cancer_Block_A_Section_2_web_summary.html')

download.file('https://cf.10xgenomics.com/samples/spatial-exp/1.1.0/V1_Breast_Cancer_Block_A_Section_1/V1_Breast_Cancer_Block_A_Section_1_filtered_feature_bc_matrix.h5','ST_Exercises/Breast_Cancer_Block_A_Section_1_filtered_feature_bc_matrix.h5')
download.file('https://cf.10xgenomics.com/samples/spatial-exp/1.1.0/V1_Breast_Cancer_Block_A_Section_1/V1_Breast_Cancer_Block_A_Section_1_spatial.tar.gz','ST_Exercises/Breast_Cancer_Block_A_Section_1_spatial.tar.gz')
download.file('https://cf.10xgenomics.com/samples/spatial-exp/1.1.0/V1_Breast_Cancer_Block_A_Section_1/V1_Breast_Cancer_Block_A_Section_1_metrics_summary.csv','ST_Exercises/Breast_Cancer_Block_A_Section_1_metrics_summary.csv')
download.file('https://cf.10xgenomics.com/samples/spatial-exp/1.1.0/V1_Breast_Cancer_Block_A_Section_1/V1_Breast_Cancer_Block_A_Section_1_web_summary.html','ST_Exercises/Breast_Cancer_Block_A_Section_1_web_summary.html')

# Lista los archivos en el directorio actual con tamaños legibles.
shell_call("ls -lh ST_Exercises")

In [ ]:
shell_call("tar -xvzf ST_Exercises/Breast_Cancer_Block_A_Section_1_spatial.tar.gz -C ST_Exercises/")
shell_call("mv ST_Exercises/spatial ST_Exercises/Spatial_Section_1")
shell_call("tar -xvzf ST_Exercises/Breast_Cancer_Block_A_Section_2_spatial.tar.gz -C ST_Exercises/")
shell_call("mv ST_Exercises/spatial ST_Exercises/Spatial_Section_2")

## Cargar el conjunto de datos

In [ ]:
samples <- list.files(
  path = "/content/ST_Exercises",
  pattern = "filtered_feature_bc_matrix.h5",
  full.names = TRUE,
  recursive = TRUE
)

imgs <- list.files(
  path = "/content/ST_Exercises",
  pattern = "tissue_lowres_image.png",
  full.names = TRUE,
  recursive = TRUE
)

spotfiles <- list.files(
  path = "/content/ST_Exercises",
  pattern = "positions_list.csv",
  full.names = TRUE,
  recursive = TRUE
)

json <- list.files(
  path = "/content/ST_Exercises",
  pattern = "scalefactors_json.json",
  full.names = TRUE,
  recursive = TRUE
)

infoTable <- tibble(samples, imgs, spotfiles, json, # Add required columns
                    sample_id = c("Breast_Cancer_Block_A_Section_1", "Breast_Cancer_Block_A_Section_2")) # Add additional column
dim(infoTable)

In [ ]:
# Carga matrices
SpatialData <- ReadVisiumData(infoTable)
SpatialData

In [ ]:
# Carga un objeto de Seurat.
# load(file = "ST_Exercises/Exercise1_dataset.RData")
# Obtiene información del conjunto de datos

# Este comando recupera el ensayo por defecto actual del objeto Seurat SpatialData
DefaultAssay(SpatialData)

# Muestra el número de genes (features) y células.
dim(SpatialData)
head(SpatialData@meta.data)

# Este comando genera una tabla de frecuencias de la columna "ids" 
# en los metadatos del objeto Seurat SpatialData.
table(SpatialData@meta.data$sample_id)


## Gráficos de control de calidad (CC)

Graficar las métricas

In [ ]:
# Crea una función de paleta de colores usando colorRampPalette del paquete grDevices, y colores del esquema "Set1" de RColorBrewer.
getPalette = colorRampPalette(brewer.pal(8, "Set1"))
color = getPalette(6)
VlnPlot(SpatialData, # Genera un violin plot (gráfico de violin)
       features = "nCount_Spatial", # Especifica la característica a graficar
       group.by = "sample_id", # Agrupa los datos según la columna "ids" de los metadatos.
       pt.size = 0.1, # Define el tamaño de los puntos en el gráfico.
       cols = color) + # Define los colores para cada grupo.
  stat_summary(fun.y=mean, geom="point", shape=95, size=15, color = "black") 
  + NoLegend()
  # Agrega una capa al gráfico con la media de la característica representada.

In [ ]:
# Gráfico de dispersión que compara la cantidad de ARN con el número de características (genes)
FeatureScatter(object = SpatialData, feature1 = "nCount_Spatial",
feature2 = "nFeature_Spatial", group.by = "sample_id", cols = color)

In [ ]:
# Carga H&E images
SpatialData <- LoadImages(SpatialData)
ImagePlot(SpatialData)

In [ ]:
# mapea las células en el tejido
MapFeatures(SpatialData, features = "nFeature_Spatial",
            image_use = "raw", override_plot_dims = TRUE) & ThemeLegendRight()

In [ ]:
MapFeaturesSummary(
  object = SpatialData,          # tu objeto semla/Seurat
  features = "nCount_Spatial",   # métrica a mostrar
  pt_size = 2.5,                 # tamaño de los puntos
  ncol = 2,                      # número de columnas en el layout
  subplot_type = "histogram"     # Tipo de grafico
)

### Alternativa de **Seurat** para graficar datos de transcriptomica espacial

In [ ]:
# crear una cópia para trabajar
SpatialData2 <- SpatialData

# Cargar archivos en un slot
SpatialData2@images[["slice1"]] <- Read10X_Image(dirname(SpatialData2@tools$Staffli@imgs[1]))
SpatialData2@images[["slice2"]] <- Read10X_Image(dirname(SpatialData2@tools$Staffli@imgs[2]))

# Vea los datos
SpatialFeaturePlot(SpatialData2, features = "nCount_Spatial") + 
                   theme(legend.position = "right")

# remove la cópia
rm(SpatialData2)

La normalización es un paso crucial del preprocesamiento en el análisis de datos de scRNA-seq. Implica ajustar los datos brutos de expresión génica para tener en cuenta las variaciones técnicas y las diferencias en la profundidad de secuenciación entre células. Estas comparaciones deberían ayudar a comprender las fortalezas y debilidades de cada método de normalización.

Método de Normalización

* SCTransform: Un método más avanzado que utiliza la regresión binomial negativa para eliminar la variabilidad técnica y estabilizar la varianza, proporcionando una normalización robusta para datos de una sola célula.

* Normalización Log: Escala los valores de expresión génica y aplica una transformación logarítmica, lo que ayuda a estabilizar la varianza y a que los datos sean más adecuados para el análisis posterior.

Eliminación de la Variabilidad Técnica

* SCTransform: Mayor robustez en la eliminación de la variabilidad técnica, lo que la hace ideal para conjuntos de datos con variación significativa.

* NormalizaciónDatos (NormalizeData): Eficaz, pero menos sofisticado que SCTransform en el manejo de la variabilidad técnica.

Complejidad y Tiempo de Ejecución

* SCTransform: Más complejo y laborioso debido a los cálculos de regresión y la transformación estabilizadora de la varianza.

* NormalizarDatos (NormalizeData): Más simple y rápido, ideal para análisis rápidos y grandes conjuntos de datos.

Manejo de Covariables

* SCTransform: Permite la inclusión de covariables (p. ej., número de genes detectados por célula) para la regresión, lo que mejora la calidad de los datos normalizados.

* NormalizarDatos (NormalizeData): Normalmente no incorpora covariables directamente en el proceso de normalización.

Estabilización de la Varianza

* SCTransform: Estabiliza la varianza, lo que hace que los datos sean más adecuados para análisis posteriores, como la agrupación en clústeres y la identificación de marcadores.

* NormalizarDatos (NormalizeData): Utiliza la transformación logarítmica para estabilizar parcialmente la varianza, pero no con la misma eficacia que SCTransform.

Flexibilidad

* SCTransform: Se centra principalmente en el manejo de datos de secuenciación de ARN de células individuales con métodos de normalización robustos.

* NormalizarDatos (NormalizeData): Flexible con diferentes métodos de normalización (p. ej., LogNormalize), lo que permite ajustes según las necesidades específicas del análisis.

Escalabilidad

* SCTransform: Puede gestionar grandes conjuntos de datos, pero puede ser más lento debido a su complejidad.

* NormalizaciónDatos  (NormalizeData): Gestiona grandes conjuntos de datos de forma eficiente gracias a su simplicidad y velocidad.


In [ ]:
# Normalizar datos usando LogNormalize con un factor de escala
SpatialData <- NormalizeData(object = SpatialData, assay = "Spatial")

In [ ]:
SpatialData <- FindVariableFeatures(SpatialData, nfeatures = 10000)

> La siguiente comparacion toma mucho tiempo **se recomienda no correr**

In [ ]:
# Normalizar datos usando SCTransform
SpatialData <- SCTransform(SpatialData, assay = "Spatial", verbose = TRUE, return.only.var.genes = FALSE)

# Comparar métodos de normalización (LogNormalize vs SCTransform)
SpatialData <- GroupCorrelation(SpatialData, group.assay = "Spatial", assay = "Spatial", layer = "data", do.plot = FALSE)
SpatialData <- GroupCorrelation(SpatialData, group.assay = "Spatial", assay = "SCT", layer = "scale.data", do.plot = FALSE)

# Generar gráficos de comparación
p1 <- GroupCorrelationPlot(SpatialData, assay = "Spatial", cor = "nCount_Spatial_cor") + ggtitle("Log Normalization")
p2 <- GroupCorrelationPlot(SpatialData, assay = "SCT", cor = "nCount_Spatial_cor") + ggtitle("SCTransform Normalization")

# Mostrar ambos gráficos lado a lado
p1 + p2

## Agrupamiento (Clustering)


In [ ]:
# Instala paquetes via GitHub
devtools::install_github("zdebruine/RcppML")
devtools::install_github("zdebruine/singlet")

In [ ]:
# carga paquete
library(singlet)

# Establecer la semilla para la reproducibilidad
set.seed(42)
SpatialData2 <- SpatialData

# OPCIONAL: subconjunto de datos para mejorar la velocidad de cálculo
SpatialData2 <- SpatialData2[VariableFeatures(SpatialData2), ]
SpatialData2 <- RunNMF(SpatialData2)

In [ ]:
# transfere las redución
SpatialData@reductions  <- SpatialData2@reductions

# Remove cópia
rm(SpatialData2)
gc()

# Transfere las columnas
k <- ncol(SpatialData@reductions$nmf@feature.loadings)
k

`RankPlot(SpatialData)` crea una figura que muestra el ranking de los factores NMF según su contribución, ayudando a seleccionar los más relevantes para interpretar patrones espaciales de expresión.

In [ ]:
# RankPlot crea una figura con un gráfico de datos de clasificación
RankPlot(SpatialData)

Veremos cómo se distribuyen espacialmente los patrones de expresión en el tejido, lo que facilita la interpretación de la heterogeneidad celular.

In [ ]:
# Ajusta las dimensiones de las gráficas que se van a generar
options(repr.plot.width=12, repr.plot.height=6)

# Primer bloque de características NMF (de 1 a 6)
MapFeatures(SpatialData,
            features = paste0("NMF_", 1:6),              # Selecciona las características NMF_1 a NMF_6
            override_plot_dims = TRUE,                   # Fuerza las dimensiones definidas arriba
            colors = viridis::magma(n = 11, direction = -1)) &  # Aplica paleta de colores 'magma' invertida
  theme(plot.title = element_blank())                   # Elimina el título del gráfico

# Segundo bloque de características NMF (de 7 a 12)
MapFeatures(SpatialData,
            features = paste0("NMF_", 7:12),             # Selecciona NMF_7 a NMF_12
            override_plot_dims = TRUE,
            colors = viridis::magma(n = 11, direction = -1)) &
  theme(plot.title = element_blank())

# Tercer bloque de características NMF (de 13 a 18)
MapFeatures(SpatialData,
            features = paste0("NMF_", 13:18),            # Selecciona NMF_13 a NMF_18
            override_plot_dims = TRUE,
            colors = viridis::magma(n = 11, direction = -1)) &
  theme(plot.title = element_blank())

# Cuarto bloque de características NMF (de 19 hasta k)
MapFeatures(SpatialData,
            features = paste0("NMF_", 19:k),             # Selecciona NMF_19 hasta NMF_k (valor máximo definido)
            override_plot_dims = TRUE,
            colors = viridis::magma(n = 11, direction = -1)) &
  theme(plot.title = element_blank())


En este paso realizamos un gráfico de cargas de características (feature loadings) derivadas de la reducción NMF. Este tipo de visualización permite identificar cuáles genes contribuyen de manera más fuerte a los primeros componentes latentes, mostrando los más relevantes en un gráfico de puntos. Con ello se facilita la interpretación biológica de los patrones espaciales y la comprensión de cómo cada gen participa en la variabilidad capturada por la NMF.

In [ ]:
# Genera un gráfico de cargas de características (feature loadings) 
# para los datos espaciales usando reducción NMF

PlotFeatureLoadings(SpatialData,
                    dims = 1:2,          # Selecciona las dos primeras dimensiones (componentes) para visualizar
                    reduction = "nmf",   # Especifica que la reducción utilizada es NMF (Non-negative Matrix Factorization)
                    nfeatures = 30,      # Número de características (genes/variables) más relevantes que se mostrarán
                    mode = "dotplot",    # Define el modo de visualización como un gráfico de puntos (dotplot)
                    fill = "darkmagenta",# Color de relleno para los puntos en el gráfico
                    pt_size = 3)         # Tamaño de los puntos en la visualización


En este paso se genera un mapa múltiple de características NMF sobre los datos espaciales. La función permite visualizar simultáneamente todos los componentes latentes (NMF_1 hasta NMF_k) proyectados en la imagen cruda del tejido. Ajustamos las dimensiones de las gráficas y el tamaño de los puntos para obtener una representación clara y uniforme, lo que facilita la interpretación de cómo cada patrón de expresión se distribuye espacialmente en el contexto tisular.

In [ ]:
# Ajusta las dimensiones de las gráficas que se van a generar
options(repr.plot.width=16, repr.plot.height=9)

# Genera un mapa múltiple de características NMF en los datos espaciales
MapMultipleFeatures(SpatialData,
            features = paste0("NMF_", 1:k),# Selecciona todas las características NMF desde 1 hasta k
            image_use = "raw",             # Usa la imagen cruda (sin procesar) como fondo de referencia
            override_plot_dims = TRUE,     # Fuerza las dimensiones definidas arriba
            pt_size = 2)                   # Define el tamaño de los puntos en la visualización


En este visualización buscamos comprender qué genes son los principales responsables de cada componente latente generado por la reducción NMF. Al visualizar las feature loadings en forma de heatmap, podemos comparar de manera clara cómo varía la contribución de los genes entre diferentes dimensiones.

In [ ]:
# Genera un mapa de calor de las cargas de características (feature loadings)
# utilizando la reducción NMF en los datos espaciales

PlotFeatureLoadings(SpatialData,
                    dims = 1:k,        # Selecciona todas las dimensiones desde 1 hasta k
                    reduction = "nmf", # Especifica que la reducción utilizada es NMF (Non-negative Matrix Factorization)
                    nfeatures = 5,     # Muestra las 5 características más relevantes por dimensión
                    mode = "heatmap",   # Define el modo de visualización como mapa de calor
                    gradient_colors = viridis::magma(n = 11,  # Aplica la paleta de colores 'magma' de la librería viridis
                                                     direction = -1)) # Invertida para resaltar mejor los gradientes


In [ ]:
# Generar UMAP
SpatialData <- RunUMAP(SpatialData, reduction = "nmf", dims = 1:k, verbose = FALSE)

In [ ]:
# Realiza la agrupación basada en grafos de Seurat
# Encuentra los vecinos más cercanos para cada célula utilizando factorización de matriz no negativa (NMF)
SpatialData <- FindNeighbors(object = SpatialData,
                      dims = 1:k, # Usa las primeras 10 dimensiones del método de reducción especificado
                      reduction = "nmf", # Indica que se utilizó NMF como método de reducción dimensional
                      verbose = FALSE) # Suprime la salida detallada

# Define una secuencia de resoluciones desde 0 hasta 2, con incrementos de 0.1
SpatialData <- FindClusters(SpatialData, resolution = seq(0, 1.2, by = 0.2),
                      verbose = FALSE)

In [ ]:
head(SpatialData@meta.data)

In [ ]:
# Visualiza los resultados de agrupamiento a diferentes resoluciones usando Clustree
clustree(SpatialData, prefix = "Spatial_snn_res.")

In [ ]:
# Crea un gráfico UMAP agrupado por "ids", sin etiquetas de agrupamiento (clúster)
DimPlot(SpatialData, reduction = "umap", label = FALSE, group.by = "sample_id",
        pt.size = 2, label.size=13)

In [ ]:
SpatialData2 <- SpatialData
SpatialData2@images[["slice1"]] <- Read10X_Image(dirname(SpatialData2@tools$Staffli@imgs[1]))
SpatialData2@images[["slice2"]] <- Read10X_Image(dirname(SpatialData2@tools$Staffli@imgs[2]))

# Grafica únicamente la lámina 3 del paciente G
# Superpone la expresión de características en coordenadas espaciales
SpatialDimPlot(SpatialData2, group.by = "Spatial_snn_res.0.4") + theme(legend.position = "right")
MapMultipleFeatures(SpatialData2,
                    features = paste0("NMF_", 1:k),
                    image_use = "raw",
                    override_plot_dims = TRUE,
                    pt_size = 2)

Ejercicio: Guardar la imagen en un archivo

Visualizar directamente la distribución de los clústeres en el espacio tisular, sin la interferencia de la imagen histológica de fondo

In [ ]:
# Grafica los clústeres sin la imagen histológica (HE) de fondo
SpatialDimPlot(SpatialData2, group.by = "Spatial_snn_res.0.4", image.alpha = 0) + 
  theme(legend.position = "right")   # Visualiza los clústeres definidos por resolución 0.4, sin transparencia de imagen

# Añade etiquetas a los clústeres en el mapa espacial
MapLabels(SpatialData, column_name = "Spatial_snn_res.0.4", ncol = 2) & 
  theme(legend.position = "right")   # Coloca la leyenda a la derecha para facilitar la lectura


## Genes expresados diferencialmente

En esta parte del análisis se identifican genes marcadores diferencialmente expresados (DEGs) entre los distintos clústeres espaciales. El filtrado por significancia estadística asegura que solo se conserven genes confiables, y el conteo final de DEGs por agrupamiento permite evaluar qué poblaciones celulares presentan perfiles moleculares más definidos o enriquecidos.

In [ ]:
devtools::install_github('immunogenomics/presto')

In [ ]:
DefaultAssay(SpatialData) # Verifica el ensayo activo del objeto Seurat SpatialData
Idents(SpatialData) <- "Spatial_snn_res.0.4" # Establece la identidad del objeto según la resolución de agrupamiento (clúster) 0.4

# Comparación de todos los agrupamientos (clústeres) entre sí
SpatialData.markers <- FindAllMarkers(object = SpatialData, # Identifica genes diferencialmente expresados (DEGs)
                                only.pos = TRUE, # Solo genes sobreexpresados en los agrupamientos (clústeres)
                                min.pct = 0.10, # El gen debe estar expresado en al menos 10% de las células
                                logfc.threshold = 0.10) # Umbral de log2 fold change

# Filtra los marcadores con p-valor ajustado < 0.05
SpatialData.markers = SpatialData.markers[which(SpatialData.markers$p_val_adj<0.05),] # Filtra los marcadores identificados para conservar solo aquellos con un valor p ajustado

# Cuenta el número de DEGs por agrupamiento (clúster)
table(SpatialData.markers[, "cluster"]) # ¿Cuántos DEGs hay por grupo?

In [ ]:
# Selecciona los 10 genes con mayor logFC por agrupamiento (clúster)
SpatialData.markers %>% group_by(cluster) %>% top_n(n = 10, wt = avg_log2FC) -> top10

# Genera un gráfico de puntos (DotPlot) para los genes seleccionados
DotPlot(SpatialData, features = unique(top10$gene),
        group.by = "Spatial_snn_res.0.4", cols = c('#b8d8d8', '#e71d36'),
        dot.scale = 6, col.min = 0) +
        theme(axis.text.x = element_text(face = "bold", color = c("black"),
        size = 8,angle = 90))

Ejercicio: Guardar la imagen en un archivo

En esa parte muestra cómo varía la expresión entre diferentes grupos, ayudando a identificar qué poblaciones de células están enriquecidas en estos genes, facilitando así la comparación directa de la intensidad y frecuencia de expresión entre grupos.

In [ ]:
# Grafica genes clave en un gráfico de densidad (ridge plot)
RidgePlot(Her2p, assay = "SCT", 
          features = c("IFI27","IFI6"), # Genes de interés
          ncol = 2, group.by = "SCT_snn_res.0.4", 
          cols = color)

In [ ]:
# Visualización UMAP y mapa de calor
# Define el gradiente de color para el mapa de calor
heatmap.colors <- c("lightgray", "mistyrose", "red", "darkred", "black")
fts <- c("SPAG6","PGM5-AS1") # Lista de genes a visualizar
# Genera los gráficos UMAP para cada gen
p.fts <- lapply(fts, function(ftr) {
  FeaturePlot(SpatialData, features = ftr, reduction = "umap", order = TRUE, cols = heatmap.colors, pt.size = )
})
# Grafica expresión transformada en coordenadas de Visium
p3 <- MapFeatures(SpatialData, features = fts, ncol = 2, cols = heatmap.colors, pt.size = 2)
# Combina todos los gráficos en una sola figura
cowplot::plot_grid(cowplot::plot_grid(plotlist = p.fts, ncol = 1), p3, ncol = 2, rel_widths = c(1, 1.3))

In [ ]:
# Gráfico de violín para expresión de EPN2 y AGR3 agrupado por clústeres
# en el objeto Seurat 'SpatialData', agrupando por la resolución de clusterización "Spatial_snn_res.0.4".
VlnPlot(SpatialData, features = c("SPAG6", "PGM5-AS1"),
        group.by = "Spatial_snn_res.0.4", # Definir variable de agrupamiento (resolución de clúster 0,4)
        pt.size = 0, # Establezca el tamaño del punto en 0 para evitar trazar puntos individuales
        ncol = 2) +  # Organiza los gráficos en dos columnas

# Superponer una estadística de resumen (media) como una barra horizontal
  stat_summary(fun.y = mean, geom = "point", shape = 95,
               size = 15, color = "black") +

# Eliminar la leyenda de la gráfica
  NoLegend()

## Visualización 3D


In [ ]:
library(plotly)
library(dplyr)
library(htmlwidgets)

Recupera las coordenadas espaciales de cada punto o célula en la imagen del tejido y las organiza por muestra. Simultáneamente, se obtiene información sobre la imagen asociada al objeto, lo que permite vincular los datos de expresión con su posición física en el tejido. Esto es fundamental en la transcriptómica espacial, que conecta los perfiles moleculares con la arquitectura tisular y facilita la interpretación biológica en su contexto espacial.

In [ ]:
# Obtiene las coordenadas espaciales de cada spot/célula en la imagen
xy_coords <- GetCoordinates(SpatialData) |>
    # Nota: se usan pxl_*_in_fullres para la imagen cruda; si las secciones están alineadas,
  # se debe usar pxl_*_in_fullres_transformed
  dplyr::select(pxl_col_in_fullres, pxl_row_in_fullres, sampleID) |>
  group_by(sampleID) |>
  group_split()   # Separa las coordenadas por muestra

image_info <- GetImageInfo(SpatialData) # Extrae información de la imagen asociada al objeto espacial


In [ ]:
# Ajusta las coordenadas espaciales de cada muestra
adjusted_coords <- do.call(bind_rows, lapply(seq_along(xy_coords), function(i) {
  xy <- xy_coords[[i]]                        # Extrae las coordenadas de la muestra i
  full_width <- image_info[i, ]$full_width    # Obtiene el ancho total de la imagen
  full_height <- image_info[i, ]$full_height  # Obtiene la altura total de la imagen
  xy <- xy |>
    mutate(x = pxl_col_in_fullres/full_width, # Normaliza la coordenada x respecto al ancho
           y = pxl_row_in_fullres/full_height) |> # Normaliza la coordenada y respecto a la altura
    # Define un valor z usando el sampleID para separar secciones en el espacio tridimensional.
    # Este ajuste permite controlar la distancia entre secciones alineadas.
    mutate(z = sampleID*0.2) |>
    dplyr::select(x, y, z)                    # Conserva solo las coordenadas ajustadas
}))


In [ ]:
# Diagrama de dispersión 3D con plotly
p <- plot_ly(adjusted_coords, x = ~x, y = ~y, z = ~z, type = "scatter3d", mode = "markers", color = SpatialData$Spatial_snn_res.0.4, size = 3)
saveWidget(as_widget(p), "SpatialData3DPlot.html")

## Términos de ontología génica (GO)

In [ ]:
# Selecciona genes diferencialmente expresados (DEGs) del clúster 6
geneList = SpatialData.markers[which(SpatialData.markers$cluster == 6), "gene"]
length(geneList)  # Muestra la cantidad de genes seleccionados

# Convierte los símbolos de genes a IDs de Entrez
columns(org.Hs.eg.db)  # Muestra las columnas disponibles para conversión en la base de anotación

symbol <- mapIds(org.Hs.eg.db,
                 keys = geneList,       # Lista de símbolos de genes a convertir
                 column = "ENTREZID",   # Convertir a IDs de Entrez
                 keytype = "SYMBOL",    # Tipo de entrada: símbolo de gen
                 multiVals = "first")   # Si hay múltiples coincidencias, toma la primera

symbol = as.vector(symbol)  # Convierte a formato vector

# Elimina genes sin ID de Entrez
geneList = symbol
geneList = geneList[which(geneList != "NA")]
length(geneList)  # Muestra la cantidad de genes después del filtrado

# Análisis de enriquecimiento de rutas con ReactomePA
x <- enrichPathway(gene = geneList,
                   organism = "human",
                   pvalueCutoff = 0.05,
                   readable = TRUE,
                   pAdjustMethod = "bonferroni")

head(as.data.frame(x))  # Muestra las primeras filas de los resultados

# Ordena los resultados por valor ajustado de p (de menor a mayor)
x@result = x@result[order(x@result$p.adjust, decreasing = FALSE),]

# Selecciona las 30 rutas más significativas
x@result = x@result[1:30,]

# Ordena rutas por cantidad de genes (de menor a mayor)
x@result = x@result[order(x@result$Count, decreasing = FALSE),]

# Guarda los resultados ordenados en un nuevo data frame
newbar.dt = x@result

# Crea un gráfico de barras con las rutas enriquecidas
cluster6_enrichpathway <- ggbarplot(newbar.dt, x = "Description", y = "Count",
          fill = "Count",         # Colorea las barras según el número de genes
          color = "white",        # Color de borde de las barras en blanco
          sort.val = "asc",       # Ordenar por valor en orden ascendente
          sort.by.groups = FALSE, # No ordenar por grupo
          x.text.angle = 90,      # Rotar etiquetas del eje x para mejor visibilidad
          xlab = "Rutas",
          ylab = "Número de genes",
          legend.title = "Recuentos",
          rotate = TRUE,
          ggtheme = theme_minimal()
) + scale_fill_continuous(low = "#bde0fe", high = "red") +
  theme(text = element_text(size = 20))

## Análisis de vías con EnrichR

In [ ]:
# Lista las bases de datos disponibles en EnrichR
listEnrichrSites()
dbs <- listEnrichrDbs()

# Ordena las bases de datos por nombre
dbs %>% dplyr::arrange(libraryName)

# Selecciona bases de datos específicas para el análisis de enriquecimiento
dbs <- c("Cancer_Cell_Line_Encyclopedia",
         "Elsevier_Pathway_Collection",
         "KEGG_2021_Human",
         "CellMarker_Augmented_2021",
         "Reactome_2016",
         "GO_Biological_Process_2018",
         "GO_Cellular_Component_2018",
         "GO_Molecular_Function_2018",
         "InterPro_Domains_2019")

# EnrichR necesita símbolos de genes (no IDs de Ensembl)
geneList = SpatialData.markers[which(SpatialData.markers$cluster == 6), "gene"]
length(geneList)  # Muestra la cantidad de genes seleccionados

# Realiza el análisis de enriquecimiento con EnrichR
enriched.Paths <- enrichr(geneList, dbs)

# Muestra resultados de las bases de datos seleccionadas
enriched.Paths[[1]]  # Cancer_Cell_Line_Encyclopedia
enriched.Paths[[2]]  # Elsevier_Pathway_Collection
enriched.Paths[[3]]  # KEGG_2021_Human
enriched.Paths[[4]]  # CellMarker_Augmented_2021
enriched.Paths[[5]]  # Reactome_2016
enriched.Paths[[6]]  # GO_Biological_Process_2018
enriched.Paths[[7]]  # GO_Cellular_Component_2018
enriched.Paths[[8]]  # GO_Molecular_Function_2018
enriched.Paths[[9]]  # InterPro_Domains_2019

# Selecciona los resultados de CellMarker_Augmented_2021
Rpath = enriched.Paths[[4]]

# Cuenta la cantidad de genes por cada ruta
Gene_Count = c()
for(x in 1:dim(Rpath)[1]){
  Gene_Count = c(Gene_Count, str_split(Rpath$Overlap[x], pattern = "/" )[[1]][1])
}

Rpath$Gene_Count = as.numeric(Gene_Count)

# Ordena por valor ajustado de p (de menor a mayor)
Rpath = Rpath[order(Rpath$Adjusted.P.value, decreasing = FALSE),]

# Filtra rutas significativas (p ajustado < 0.05)
Rpath = Rpath[Rpath$Adjusted.P.value < 0.05,]

# Selecciona las 20 rutas principales
Rpath = Rpath[1:20,]

# Ordena rutas por número de genes (de mayor a menor)
Rpath = Rpath[order(Rpath$Gene_Count, decreasing = TRUE),]

# Crea un gráfico de barras de rutas enriquecidas
cluster6_enrichR <- ggbarplot(Rpath, x = "Term", y = "Gene_Count",
          fill = "Gene_Count",         # Colorea las barras según número de genes
          color = "white",             # Color de borde blanco
          sort.val = "asc",            # Ordena por valor en orden ascendente
          sort.by.groups = FALSE,
          x.text.angle = 90,           # Gira etiquetas para visibilidad
          xlab = "Rutas",
          ylab = "Número de genes",
          legend.title = "Recuentos",
          rotate = TRUE,
          ggtheme = theme_minimal()
) + scale_fill_continuous(low = "#bde0fe", high = "red") +
  theme(text = element_text(size = 18))

# Muestra el gráfico
cluster6_enrichR

## Deconvolución de tipos de células

La deconvolución es una técnica computacional que se utiliza para inferir y cuantificar las proporciones de diferentes tipos celulares dentro de una población celular mixta. Dado que los datos de scRNA-seq suelen provenir de tejidos complejos que contienen múltiples tipos celulares, la deconvolución ayuda a identificar y aislar los perfiles de expresión génica específicos de cada tipo celular dentro de la mezcla.

¿Por qué es importante la deconvolución?

* Identificación de tipos celulares: La deconvolución permite a los investigadores determinar la presencia y abundancia de varios tipos celulares en una muestra de tejido heterogénea.
* Comprender la composición celular: Ayuda a dilucidar la composición celular de los tejidos, revelando cómo los diferentes tipos celulares contribuyen a los procesos biológicos y las enfermedades.
* Mejorar la interpretación de datos: Al separar las señales de expresión génica de los diferentes tipos celulares, la deconvolución mejora la precisión y la interpretabilidad de los datos de scRNA-seq, lo que permite obtener información biológica más precisa.


In [ ]:
download.file('https://github.com/integrativebioinformatics/scNotebooks/blob/main/scNotebooks-Resources/scRNASeq_pac2_processed.rds', 'ST_Exercises/scRNASeq_pac2_processed.rds')

In [ ]:
# Carga el dataset de scRNA-seq
SC.data = readRDS("ST_Exercises/scRNASeq_pac2_processed.rds")
SC.data <-UpdateSeuratObject(SC.data)

# Establece identidades de células según tipo celular
Idents(SC.data) = "cellType"

# Visualización UMAP de tipos celulares
DimPlot(object = SC.data, reduction = 'umap', label = TRUE, label.size = 6,
        group.by = "cellType", pt.size = 1.5)

# Identifica DEGs entre todos los agrupamientos (clústeres)
SC.markers <- FindAllMarkers(object = SC.data, only.pos = TRUE, min.pct = 0.10, logfc.threshold = 0.10)
# min.pct = 0.10: al menos 10% de las células debe expresar el gen
# logfc.threshold = 0.10: umbral mínimo de log2 fold change para considerar el gen como marcador

SC.markers = SC.markers[which(SC.markers$p_val_adj < 0.05 & SC.markers$avg_log2FC > 0.5),]
# SC.markers$p_val_adj < 0.05: This condition selects markers with an adjusted p-value less than 0.05, meaning they are statistically significant.
# SC.markers$avg_log2FC > 0.5: This condition selects markers with an average log2 fold change greater than 0.5

# Cuenta el número de DEGs por agrupamiento (clúster)
table(SC.markers$cluster)

DefaultAssay(SC.data) # devuelve el nombre del ensayo actualmente establecido como predeterminado para el objeto Seurat

## Flujo en Tubería (Pipeline) de deconvolución

In [ ]:
#ti <- Sys.time()
DefaultAssay(SpatialData) <- "Spatial"

# Predicción de proporciones de tipos de células
SpatialData <- RunNNLS(object = SpatialData,
                      singlecell_object = SC.data,
                      groups = "cellType")

In [ ]:
# Comprobar los tipos de celdas disponibles
rownames(SpatialData)

In [ ]:
# Carga H&E images
SpatialData <- SpatialData |>
  LoadImages()

In [ ]:
# Trazar múltiples características (features)
MapMultipleFeatures(SpatialData,
                    image_use = "raw",
                    pt_size = 2, max_cutoff = 0.95,
                    override_plot_dims = TRUE,
                    features = rownames(SpatialData)) +
  plot_layout(guides = "collect")

MapMultipleFeatures(SpatialData,
                    pt_size = 2, max_cutoff = 0.95,
                    override_plot_dims = TRUE,
                    features = rownames(SpatialData)) +
  plot_layout(guides = "collect")

## Cell type co-localization

La co-localización de tipos celulares se refiere al análisis de cómo diferentes poblaciones celulares se distribuyen y aparecen juntas en regiones específicas del tejido. En transcriptómica espacial, este paso permite identificar patrones de proximidad o interacción entre tipos celulares, revelando posibles relaciones funcionales, comunicación celular o microambientes relevantes para procesos biológicos y enfermedades.

In [ ]:
library(pheatmap)

# Extrae la matriz de expresión de SpatialData y calcula la correlación entre genes/tipos celulares
cor_matrix <- FetchData(SpatialData, rownames(SpatialData)) |>
  mutate_all(~ if_else(.x<0.1, 0, .x)) |>  # Filtra valores muy bajos, estableciéndolos en 0
  cor()                                    # Calcula la matriz de correlación

diag(cor_matrix) <- NA                     # Elimina la diagonal (autocorrelaciones) para mayor claridad
max_val <- max(cor_matrix, na.rm = T)      # Obtiene el valor máximo de correlación

# Define la paleta de colores para el heatmap (rojo-amarillo-azul invertida, con blanco en el centro)
cols <- RColorBrewer::brewer.pal(7, "RdYlBu") |> rev(); cols[4] <- "white"

# Ajusta las dimensiones de la figura
options(repr.plot.width=6, repr.plot.height=6)

# Genera el heatmap de correlaciones entre tipos celulares dentro de los spots
pheatmap::pheatmap(cor_matrix,
                   breaks = seq(-max_val, max_val, length.out = 100), # Rango de valores de correlación
                   color=colorRampPalette(cols)(100),                 # Gradiente de colores
                   cellwidth = 14, cellheight = 14,                   # Tamaño de las celdas
                   treeheight_col = 10, treeheight_row = 10,          # Altura de los dendrogramas
                   main = "Cell type correlation\nwithin spots")      # Título del gráfico


In [ ]:
# Aplica la factorización matricial no negativa (NMF) sobre los datos espaciales
nmf_data <- FetchData(SpatialData, rownames(SpatialData)) |>   # Extrae la matriz de expresión del objeto espacial
  RcppML::nmf(k = 10, verbose = T)                             # Ejecuta NMF con k = 10 componentes latentes, mostrando mensajes en consola


In [ ]:
# Convierte la matriz H del resultado NMF en un data.frame
nmf_data_h <- nmf_data@h |> as.data.frame()

# Asigna nombres de fila a los factores (Factor_1 a Factor_10)
rownames(nmf_data_h) <- paste0("Factor_", 1:10)

# Asigna nombres de columna correspondientes a las células/spots del objeto espacial
colnames(nmf_data_h) <- rownames(SpatialData)

# Normaliza los valores de cada columna dividiendo por el valor máximo ( 0 y 1)
nmf_data_h <- nmf_data_h |>
  mutate_at(colnames(nmf_data_h),
            ~(scale(., center = FALSE, scale = max(., na.rm = TRUE)/1)))

# Crea una columna Factor con los nombres de fila, manteniendo el orden de los factores
nmf_data_h$Factor <- rownames(nmf_data_h) |>
  factor(levels = paste0("Factor_", 1:10))

# Reorganiza los datos en formato largo (long format):
# cada fila representa el peso (Weight) de un factor en una célula específica
nmf_data_h_df <- nmf_data_h |>
  tidyr::pivot_longer(cols = all_of(rownames(SpatialData)),
                      names_to = "Cell",
                      values_to = "Weight")


In [ ]:
# Bubble chart
ggplot(nmf_data_h_df, aes(x=Factor, y=Cell, size=Weight, color=Weight)) +
  geom_point() +
  labs(title="Cell type contribution", x="Factor", y = "Cell type",
       color = "", size = "Scaled weight") +
  scale_color_viridis_c(direction = -1, option = "magma") +
  theme_bw() +
  theme(axis.text.x = element_text(angle=45, hjust=1),
        panel.grid = element_blank())